In [1]:
import pandas as pd
import numpy as np

In [4]:
df_all = pd.read_parquet("all_merged_repaired.parquet")

In [5]:
print(df_all.columns.tolist())

['date', 'security_id', 'report_date', 'ticker', 'company_id', 'announce_date', 'fiscal_year', 'fiscal_quarter', 'too_many_quarters_flag', 'too_few_quarters_flag', 'transition_flag', 'total_assets', 'cash_and_equivalents', 'inventory', 'receivables', 'net_ppe', 'total_assets_dnu_flag', 'cash_and_equivalents_dnu_flag', 'inventory_dnu_flag', 'receivables_dnu_flag', 'net_ppe_dnu_flag', 'total_assets_imputed_flag', 'cash_and_equivalents_imputed_flag', 'inventory_imputed_flag', 'receivables_imputed_flag', 'net_ppe_imputed_flag', 'total_liabilities', 'current_liabilities', 'long_term_debt', 'short_term_debt', 'book_equity', 'common_equity', 'total_liabilities_imputed_flag', 'current_liabilities_imputed_flag', 'long_term_debt_imputed_flag', 'short_term_debt_imputed_flag', 'book_equity_imputed_flag', 'common_equity_imputed_flag', 'total_liabilities_dnu_flag', 'current_liabilities_dnu_flag', 'long_term_debt_dnu_flag', 'short_term_debt_dnu_flag', 'book_equity_dnu_flag', 'common_equity_dnu_flag',

In [26]:
# ============ Quarterly Alpha 需要的列(分组) ============

# A. 键 + 元数据 + 分类
KEYS_META = [
    'date','security_id','ticker','company_id','report_date','announce_date',
    'fiscal_year','fiscal_quarter',
    'gics_sector','gics_industry','gics_subindustry',
]
# 季度结构 / PIT 质检旗标(用于剔坏季度/首季)
QUARTER_QC = [
    'too_many_quarters_flag','too_few_quarters_flag','transition_flag',
    'announce_date_filled_flag','end_forwardfill_flag',
]
# B. 季度会计原始变量(26 个,全部需要)
ACCT = [
    'total_assets','cash_and_equivalents','inventory','receivables','net_ppe',
    'total_liabilities','current_liabilities','long_term_debt','short_term_debt',
    'book_equity','common_equity','common_shares_outstanding_mn','dividend_per_share',
    'operating_cash_flow_ytd','capex_ytd','ocf_increment','capex_increment',
    'revenue','cogs','gross_profit','operating_income','interest_expense',
    'income_taxes','net_income','eps_basic','eps_diluted',
]
# C. 季度 consensus(含 statistic_date 快照键 + estimate_high/low)
CONSENSUS = [
    'consensus_statistic_date','consensus_forecast_period_end',
    'consensus_analyst_count','consensus_numup','consensus_numdown',
    'consensus_eps_mean','consensus_eps_median','consensus_estimate_dispersion',
    'consensus_estimate_high','consensus_estimate_low',
]
# C-opt. 队友加的 PIT 辅助列(days_to_period_end / actual_eps_period_end 已删)
CONSENSUS_OPT = [
    'consensus_days_since_actual_eps',
]
# D. Earnings actuals 事件(Surprise_Direction 用;announce_time 已删)
ACTUALS = [
    'actuals_period_end','actuals_announce_date','actuals_actual_eps',
    'actuals_announce_date_missing',  # 非 _flag 结尾,单列
]

# ---- 显式排除:所有 _imputed_flag + actuals_eps_outlier_flag ----
EXCLUDE = {'actuals_eps_outlier_flag'}
def _keep_flag(c):
    return not c.endswith('_imputed_flag') and c not in EXCLUDE

# ---- 质检旗标:按前缀自动收集,只收 quarterly 相关,不碰 rec_/tp_/industry_/sector_ ----
ACCT_FLAGS = [c for c in ALL_MERGED_COLUMNS
              if c.endswith('_flag') and any(c.startswith(v+'_') for v in ACCT) and _keep_flag(c)]
OCF_CAPEX_FLAGS = [c for c in ['ocf_dnu_flag','capex_dnu_flag','ocf_structural_missing_flag',
                   'capex_structural_missing_flag','ocf_imputed_flag','capex_imputed_flag'] if _keep_flag(c)]
CONSENSUS_FLAGS = [c for c in ALL_MERGED_COLUMNS if c.startswith('consensus_') and c.endswith('_flag') and _keep_flag(c)]
ACTUALS_FLAGS  = [c for c in ALL_MERGED_COLUMNS if c.startswith('actuals_') and c.endswith('_flag') and _keep_flag(c)]

QC_FLAGS = QUARTER_QC + sorted(set(ACCT_FLAGS + OCF_CAPEX_FLAGS)) + CONSENSUS_FLAGS + ACTUALS_FLAGS

# ---- 汇总(去重、保序) ----
GROUPS = [('A 键/元数据/分类',KEYS_META),('B 会计',ACCT),('C consensus',CONSENSUS),
          ('C-opt consensus辅助',CONSENSUS_OPT),('D actuals事件',ACTUALS),
          ('E 质检旗标',QC_FLAGS)]

quarterly_cols, seen = [], set()
for _,g in GROUPS:
    for c in g:
        if c not in seen:
            seen.add(c); quarterly_cols.append(c)

# ---- 自检:确认每一列都真实存在于 df_all ----
missing = [c for c in quarterly_cols if c not in ALL_MERGED_COLUMNS]
print("=== 自检 ===")
print("df_all 总列数:", len(ALL_MERGED_COLUMNS))
print("选中列数    :", len(quarterly_cols))
print("不存在的列  :", missing if missing else "无 ✅ 全部存在")
print()
for name,g in GROUPS:
    g2=[c for c in g if c in ALL_MERGED_COLUMNS]
    print(f"[{name}] {len(g2)} 列")
print()
print("=== quarterly_cols ===")
print(quarterly_cols)

=== 自检 ===
df_all 总列数: 187
选中列数    : 88
不存在的列  : 无 ✅ 全部存在

[A 键/元数据/分类] 11 列
[B 会计] 26 列
[C consensus] 10 列
[C-opt consensus辅助] 1 列
[D actuals事件] 4 列
[E 质检旗标] 36 列

=== quarterly_cols ===
['date', 'security_id', 'ticker', 'company_id', 'report_date', 'announce_date', 'fiscal_year', 'fiscal_quarter', 'gics_sector', 'gics_industry', 'gics_subindustry', 'total_assets', 'cash_and_equivalents', 'inventory', 'receivables', 'net_ppe', 'total_liabilities', 'current_liabilities', 'long_term_debt', 'short_term_debt', 'book_equity', 'common_equity', 'common_shares_outstanding_mn', 'dividend_per_share', 'operating_cash_flow_ytd', 'capex_ytd', 'ocf_increment', 'capex_increment', 'revenue', 'cogs', 'gross_profit', 'operating_income', 'interest_expense', 'income_taxes', 'net_income', 'eps_basic', 'eps_diluted', 'consensus_statistic_date', 'consensus_forecast_period_end', 'consensus_analyst_count', 'consensus_numup', 'consensus_numdown', 'consensus_eps_mean', 'consensus_eps_median', 'consensus_estimat

In [27]:
df_quarterly = df_all[quarterly_cols].copy()

In [28]:
df_quarterly

,date,security_id,ticker,company_id,report_date,announce_date,fiscal_year,fiscal_quarter,gics_sector,gics_industry,...,short_term_debt_dnu_flag,total_assets_dnu_flag,total_liabilities_dnu_flag,consensus_stale_consensus_flag,consensus_single_analyst_flag,consensus_eps_outlier_flag,consensus_forward_looking_flag,actuals_early_announcement_flag,actuals_long_gap_flag,actuals_missing_eps_flag
0,2011-01-20,10001,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,False,False,False,True,True,False,False,False,False,True
1,2011-01-21,10001,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,False,False,False,True,True,False,False,False,False,True
2,2011-01-24,10001,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,False,False,False,True,True,False,False,False,False,True
3,2011-01-25,10001,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,False,False,False,True,True,False,False,False,False,True
4,2011-01-26,10001,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,False,False,False,True,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4133562,2024-12-24,93436,TSLA,184996,2024-09-30,2024-10-23,2024.0,3.0,25,251020,...,False,False,False,False,False,False,False,False,False,False
4133563,2024-12-26,93436,TSLA,184996,2024-09-30,2024-10-23,2024.0,3.0,25,251020,...,False,False,False,False,False,False,False,False,False,False
4133564,2024-12-27,93436,TSLA,184996,2024-09-30,2024-10-23,2024.0,3.0,25,251020,...,False,False,False,False,False,False,False,False,False,False
4133565,2024-12-30,93436,TSLA,184996,2024-09-30,2024-10-23,2024.0,3.0,25,251020,...,False,False,False,False,False,False,False,False,False,False


In [29]:
df_quarterly.columns.tolist()

['date',
 'security_id',
 'ticker',
 'company_id',
 'report_date',
 'announce_date',
 'fiscal_year',
 'fiscal_quarter',
 'gics_sector',
 'gics_industry',
 'gics_subindustry',
 'total_assets',
 'cash_and_equivalents',
 'inventory',
 'receivables',
 'net_ppe',
 'total_liabilities',
 'current_liabilities',
 'long_term_debt',
 'short_term_debt',
 'book_equity',
 'common_equity',
 'common_shares_outstanding_mn',
 'dividend_per_share',
 'operating_cash_flow_ytd',
 'capex_ytd',
 'ocf_increment',
 'capex_increment',
 'revenue',
 'cogs',
 'gross_profit',
 'operating_income',
 'interest_expense',
 'income_taxes',
 'net_income',
 'eps_basic',
 'eps_diluted',
 'consensus_statistic_date',
 'consensus_forecast_period_end',
 'consensus_analyst_count',
 'consensus_numup',
 'consensus_numdown',
 'consensus_eps_mean',
 'consensus_eps_median',
 'consensus_estimate_dispersion',
 'consensus_estimate_high',
 'consensus_estimate_low',
 'consensus_days_since_actual_eps',
 'actuals_period_end',
 'actuals_annou

In [44]:
import numpy as np
import pandas as pd

REBAL_MD = [(3, 31), (6, 30), (9, 30), (12, 31)]


def build_rebalance_panel(df):
    df = df.sort_values(['security_id', 'date']).reset_index(drop=True)

    # 统一所有 datetime 列精度到 [ns](kind=='M' 覆盖 us/ms/ns)
    for c in df.columns:
        if df[c].dtype.kind == 'M':
            df[c] = df[c].astype('datetime64[ns]')

    consensus_cols = [c for c in df.columns if c.startswith('consensus_')]
    actuals_cols   = [c for c in df.columns if c.startswith('actuals_')]
    fmt_cols       = [c for c in df.columns
                      if c not in consensus_cols + actuals_cols + ['security_id', 'date']]

    # 调仓日网格(强制 [ns]),限制在每只票数据区间内
    yrs = range(df['date'].dt.year.min(), df['date'].dt.year.max() + 1)
    all_rebal = pd.to_datetime([f'{y}-{m:02d}-{d:02d}' for y in yrs
                                for m, d in REBAL_MD]).astype('datetime64[ns]')
    rng = df.groupby('security_id')['date'].agg(['min', 'max'])
    grid = (rng.assign(key=1).reset_index()
              .merge(pd.DataFrame({'rebalance_date': all_rebal, 'key': 1}), on='key')
              .query('min <= rebalance_date <= max')
              [['security_id', 'rebalance_date']]
              .sort_values(['security_id', 'rebalance_date']).reset_index(drop=True))

    # FMT: announce_date <= 调仓日 最近一条
    fmt = (df[['security_id'] + fmt_cols]
           .dropna(subset=['announce_date'])
           .sort_values('announce_date')
           .drop_duplicates(['security_id', 'announce_date'], keep='last'))
    res = pd.merge_asof(grid.sort_values('rebalance_date'), fmt.sort_values('announce_date'),
                        by='security_id', left_on='rebalance_date', right_on='announce_date',
                        direction='backward')

    # actuals: actuals_announce_date <= 调仓日 最近一条
    act = (df[['security_id'] + actuals_cols]
           .dropna(subset=['actuals_announce_date'])
           .sort_values('actuals_announce_date')
           .drop_duplicates(['security_id', 'actuals_announce_date'], keep='last'))
    res = pd.merge_asof(res.sort_values('rebalance_date'), act.sort_values('actuals_announce_date'),
                        by='security_id', left_on='rebalance_date', right_on='actuals_announce_date',
                        direction='backward')

    # consensus: 期P 相同 且 statistic_date < A 最后一条
    cons = (df[['security_id'] + consensus_cols]
            .dropna(subset=['consensus_forecast_period_end', 'consensus_statistic_date'])
            .drop_duplicates(['security_id', 'consensus_forecast_period_end',
                              'consensus_statistic_date'], keep='last').copy())
    cons['_pe'] = cons['consensus_forecast_period_end'].astype('datetime64[ns]')

    left = res[['security_id', 'rebalance_date', 'actuals_period_end', 'actuals_announce_date']].copy()
    left['_pe'] = left['actuals_period_end'].astype('datetime64[ns]')
    left_ok = left.dropna(subset=['actuals_period_end', 'actuals_announce_date'])

    cm = pd.merge_asof(
        left_ok.sort_values('actuals_announce_date'),
        cons.sort_values('consensus_statistic_date'),
        by=['security_id', '_pe'],
        left_on='actuals_announce_date', right_on='consensus_statistic_date',
        direction='backward', allow_exact_matches=True)

    res = res.merge(cm[['security_id', 'rebalance_date'] + consensus_cols],
                    on=['security_id', 'rebalance_date'], how='left')
    return res.sort_values(['security_id', 'rebalance_date']).reset_index(drop=True)

In [45]:
panel = build_rebalance_panel(df_quarterly)
print(len(panel)); panel.head()

66349


,security_id,rebalance_date,ticker,company_id,report_date,announce_date,fiscal_year,fiscal_quarter,gics_sector,gics_industry,...,consensus_eps_mean,consensus_eps_median,consensus_estimate_dispersion,consensus_estimate_high,consensus_estimate_low,consensus_days_since_actual_eps,consensus_stale_consensus_flag,consensus_single_analyst_flag,consensus_eps_outlier_flag,consensus_forward_looking_flag
0,10001,2011-03-31,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,2011-06-30,EGAS,012994,2011-03-31,2011-05-11,2011.0,1.0,55,551020,...,0.48,0.48,0.0,0.48,0.48,7.0,True,True,False,False
2,10001,2011-09-30,EGAS,012994,2011-06-30,2011-08-12,2011.0,2.0,55,551020,...,0.01,0.01,0.0,0.01,0.01,5.0,True,True,False,False
3,10001,2011-12-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.0,-0.11,-0.11,2.0,True,True,False,False
4,10001,2012-03-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.0,-0.11,-0.11,2.0,True,True,False,False


In [46]:
panel

,security_id,rebalance_date,ticker,company_id,report_date,announce_date,fiscal_year,fiscal_quarter,gics_sector,gics_industry,...,consensus_eps_mean,consensus_eps_median,consensus_estimate_dispersion,consensus_estimate_high,consensus_estimate_low,consensus_days_since_actual_eps,consensus_stale_consensus_flag,consensus_single_analyst_flag,consensus_eps_outlier_flag,consensus_forward_looking_flag
0,10001,2011-03-31,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,2011-06-30,EGAS,012994,2011-03-31,2011-05-11,2011.0,1.0,55,551020,...,0.48,0.48,0.00,0.48,0.48,7.0,True,True,False,False
2,10001,2011-09-30,EGAS,012994,2011-06-30,2011-08-12,2011.0,2.0,55,551020,...,0.01,0.01,0.00,0.01,0.01,5.0,True,True,False,False
3,10001,2011-12-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.00,-0.11,-0.11,2.0,True,True,False,False
4,10001,2012-03-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.00,-0.11,-0.11,2.0,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66344,93436,2023-12-31,TSLA,184996,2023-09-30,2023-10-18,2023.0,3.0,25,251020,...,0.80,0.80,0.09,1.02,0.65,0.0,False,False,False,False
66345,93436,2024-03-31,TSLA,184996,2023-12-31,2024-01-24,2023.0,4.0,25,251020,...,0.74,0.74,0.08,0.92,0.55,21.0,True,False,False,False
66346,93436,2024-06-30,TSLA,184996,2024-03-31,2024-04-23,2024.0,1.0,25,251020,...,0.51,0.45,0.17,0.87,0.20,22.0,True,False,False,False
66347,93436,2024-09-30,TSLA,184996,2024-06-30,2024-07-23,2024.0,2.0,25,251020,...,0.62,0.62,0.11,0.87,0.42,22.0,True,False,False,False


In [47]:
panel.to_parquet("fmt_quarterly_panel.parquet", index=False)

In [2]:
import pandas as pd
pd.read_parquet("D:\icarus_alpha_calculation\data_separation\daily_clean.parquet").head(10)

,date,security_id,ticker,company_id,gics_sector,gics_industry,gics_subindustry,equity_ticker,equity_company_name_pit,equity_share_class_code,...,equity_adj_open,equity_adj_high,equity_adj_low,equity_adj_close,equity_adj_volume,equity_total_return,equity_price_return_ex_div,equity_shares_outstanding_k,equity_price_adj_factor,equity_share_adj_factor
0,2011-01-20,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.79,10.7900,10.6800,10.7600,20600.0,0.002796,0.002796,7834.0,1.0,1.0
1,2011-01-21,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.81,10.8100,10.5500,10.6699,21100.0,-0.008374,-0.008374,7834.0,1.0,1.0
2,2011-01-24,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.74,10.8800,10.7201,10.7700,20100.0,0.009382,0.009382,7834.0,1.0,1.0
3,2011-01-25,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.81,10.8699,10.7850,10.8699,22000.0,0.009276,0.009276,7834.0,1.0,1.0
4,2011-01-26,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.91,10.9200,10.8000,10.9200,30300.0,0.004609,0.004609,7834.0,1.0,1.0
5,2011-01-27,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.90,10.9200,10.8000,10.8000,23500.0,-0.010989,-0.010989,7834.0,1.0,1.0
6,2011-01-28,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.90,10.9000,10.7200,10.7400,29700.0,-0.005556,-0.005556,7834.0,1.0,1.0
7,2011-01-31,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.83,10.9000,10.7200,10.7800,25000.0,0.003724,0.003724,7834.0,1.0,1.0
8,2011-02-01,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.83,10.8440,10.7500,10.7700,25800.0,-0.000928,-0.000928,7834.0,1.0,1.0
9,2011-02-02,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.84,10.8800,10.7500,10.7600,17000.0,-0.000929,-0.000929,7834.0,1.0,1.0
